# 02 — Stage-2 Progressive 512px -> 768px ResViT-Pro Training (Google Colab GPU / TPU)
Trains `ResViTPro` across the Two-Phase Curriculum on `stage2_grading_splits.csv` (3,395 clean high-resolution APTOS 2019 scans):
- **Phase A (Epochs 1–15, 512×512):** Stable representation learning with `CORALOrdinalFocalLoss` + Referable-DR BCE + Continuous Smooth-L1.
- **Phase B (Epochs 16–30, 768×768):** Unlocks learnable `GeM` Pooling ($p$-norm), Stride-4 FPN micro-lesion features, and full Differentiable `SoftQWKLoss` weight (`0.75`).
- **Class Imbalance Mitigation:** Uses `EffectiveNumberSampler` + `WeightedRandomSampler` so Grade 1 (Mild DR) and Grade 3 (Severe DR) are balanced in every batch.

In [ ]:
# 1. Mount Google Drive, Clone Repo & Ensure Preprocessed Data is Unpacked
import os, sys, shutil, tarfile, zipfile
from pathlib import Path
import subprocess
import torch
import torch.nn.functional as F

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/Retinopathy')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dprincepiriyan/Retinopathy.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    LOCAL_DATA_ROOT = Path('/content/data/processed')
    if not (LOCAL_DATA_ROOT / 'stage2_grading_splits.csv').exists():
        drive_root = Path('/content/drive/MyDrive')
        candidates = list(drive_root.rglob('retinopathy_processed.tar.gz')) + list(drive_root.rglob('retinopathy_processed.zip'))
        if not candidates:
            raise FileNotFoundError(
                'Could not find retinopathy_processed.tar.gz in Google Drive! '
                'Please upload retinopathy_processed.tar.gz to your Google Drive first.'
            )
        archive_path = candidates[0]
        print(f'Extracting {archive_path} to /content/data/ ...')
        Path('/content/data').mkdir(parents=True, exist_ok=True)
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as zf:
                zf.extractall('/content/data')
        else:
            with tarfile.open(archive_path, 'r:gz') as tf:
                tf.extractall('/content/data')
        print('Extraction complete!')
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Retinopathy_Checkpoints')
else:
    REPO_DIR = Path.cwd().resolve()
    if REPO_DIR.name == 'colab':
        REPO_DIR = REPO_DIR.parent
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    LOCAL_DATA_ROOT = REPO_DIR / 'data' / 'processed'
    DRIVE_CKPT_DIR = REPO_DIR / 'checkpoints'

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT_DIR = REPO_DIR / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

from src.utils.tpu_accelerator import ColabTPUAccelerator
accel = ColabTPUAccelerator(prefer_tpu=True)
print(f'Active Accelerator: {accel.info.device_name} | Backend: {accel.info.backend} | Precision: {accel.info.amp_dtype}')

In [ ]:
# 2. Helper to Build Resolution-Specific Stage-2 DataLoaders with Class-Balanced Sampling
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from src.data.datasets import PreprocessedManifestDataset
from src.data.samplers import EffectiveNumberSampler, CurriculumResolutionScheduler

MANIFEST_CSV = str(LOCAL_DATA_ROOT / 'stage2_grading_splits.csv')
VAL_FOLD = 4

def build_stage2_loaders(resolution: int, batch_size: int):
    train_ds = PreprocessedManifestDataset(
        manifest_csv=MANIFEST_CSV,
        split='train_val',
        exclude_fold=VAL_FOLD,
        resolution=resolution,
        data_root_override=str(LOCAL_DATA_ROOT),
        augment=True,
    )
    val_ds = PreprocessedManifestDataset(
        manifest_csv=MANIFEST_CSV,
        split='train_val',
        only_fold=VAL_FOLD,
        resolution=resolution,
        data_root_override=str(LOCAL_DATA_ROOT),
        augment=False,
    )
    test_ds = PreprocessedManifestDataset(
        manifest_csv=MANIFEST_CSV,
        split='test',
        resolution=resolution,
        data_root_override=str(LOCAL_DATA_ROOT),
        augment=False,
    )

    train_labels = np.array([int(r['dr_grade']) for r in train_ds.rows], dtype=np.int64)
    eff_sampler = EffectiveNumberSampler(beta=0.999)
    sample_probs = eff_sampler.compute_sample_probabilities(train_labels, num_classes=5)
    weighted_sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_probs).double(),
        num_samples=len(train_ds),
        replacement=True,
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=weighted_sampler, num_workers=2,
        pin_memory=(accel.info.backend == 'cuda'), drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=2,
        pin_memory=(accel.info.backend == 'cuda'), drop_last=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=2,
        pin_memory=(accel.info.backend == 'cuda'), drop_last=False
    )
    return train_loader, val_loader, test_loader

train_loader_512, val_loader_512, _ = build_stage2_loaders(resolution=512, batch_size=16)
train_loader_768, val_loader_768, test_loader_768 = build_stage2_loaders(resolution=768, batch_size=8)
print(f'Stage-2 Train: {len(train_loader_512.dataset)} | Val: {len(val_loader_512.dataset)} | Test: {len(test_loader_768.dataset)}')

In [ ]:
# 3. Initialize ResViT-Pro Architecture, Tri-Head Losses & Curriculum Scheduler
from src.tier3_grading import ResViTPro
from src.losses import CORALOrdinalFocalLoss, SoftQWKLoss
from src.utils.metrics import compute_clinical_metrics

PHASE_A_EPOCHS = 15
PHASE_B_EPOCHS = 15

curriculum = CurriculumResolutionScheduler(
    phase_a_epochs=PHASE_A_EPOCHS,
    phase_b_epochs=PHASE_B_EPOCHS,
    phase_a_res=512,
    phase_b_res=768,
    phase_a_lr=2.0e-4,
    phase_b_lr=6.0e-5,
)

model = ResViTPro(fpn_dim=64, num_classes=5, num_heads=4, gem_p=3.0, dropout=0.25).to(accel.device)
coral_loss_fn = CORALOrdinalFocalLoss(num_classes=5).to(accel.device)
qwk_loss_fn = SoftQWKLoss(num_classes=5).to(accel.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2.0e-4, weight_decay=1e-2)

def evaluate_stage2(loader, qwk_weight: float = 0.75):
    model.eval()
    all_probs, all_cont, all_targets, all_ref_probs = [], [], [], []
    total_loss, n_batches = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            x = batch['image'].to(accel.device)
            y_grades = batch['dr_grade'].to(accel.device)
            y_ref = batch['referable_dr'].to(accel.device)

            with accel.autocast_context(enabled=True):
                out = model(x)
                tri = out['tri_head']
                loss_ord = coral_loss_fn(tri.ordinal_logits, y_grades)
                loss_qwk = qwk_loss_fn(tri.class_probs, y_grades, is_probs=True)
                loss_ref = F.binary_cross_entropy_with_logits(tri.referable_logit, y_ref)
                loss_cont = F.smooth_l1_loss(tri.continuous_score.squeeze(1), y_grades.float())
                loss = loss_ord + qwk_weight * loss_qwk + 0.50 * loss_ref + 0.40 * loss_cont

            total_loss += float(loss.detach().cpu().item())
            n_batches += 1
            all_probs.append(tri.class_probs.detach().float().cpu().numpy())
            all_cont.append(tri.continuous_score.squeeze(1).detach().float().cpu().numpy())
            all_ref_probs.append(tri.referable_prob.squeeze(1).detach().float().cpu().numpy())
            all_targets.append(y_grades.detach().cpu().numpy())

    probs_arr = np.concatenate(all_probs, axis=0)
    cont_arr = np.concatenate(all_cont, axis=0)
    ref_arr = np.concatenate(all_ref_probs, axis=0)
    targets_arr = np.concatenate(all_targets, axis=0)
    preds_arr = np.argmax(probs_arr, axis=1)

    clin = compute_clinical_metrics(targets_arr, preds_arr, probs_arr)
    clin['loss'] = round(total_loss / max(n_batches, 1), 4)
    return clin, {
        'probs': probs_arr,
        'continuous_scores': cont_arr,
        'referable_probs': ref_arr,
        'targets': targets_arr,
    }

In [ ]:
# 4. Execute Two-Phase Progressive Training (Phase A: 512px -> Phase B: 768px)
import time

best_val_qwk = -1.0
best_ckpt_local = LOCAL_CKPT_DIR / 'stage2_resvit_pro_768_best.pth'
best_ckpt_drive = DRIVE_CKPT_DIR / 'stage2_resvit_pro_768_best.pth'

print(f'Starting Stage-2 Two-Phase Training ({curriculum.total_epochs} total epochs)...')
for epoch in range(1, curriculum.total_epochs + 1):
    t0 = time.time()
    cfg = curriculum.get_epoch_config(epoch)

    # Update learning rate and GeM learnability according to curriculum
    for param_group in optimizer.param_groups:
        param_group['lr'] = cfg.learning_rate
    model.gem_pool.p.requires_grad_(cfg.enable_gem_learning)

    active_train_loader = train_loader_512 if cfg.resolution == 512 else train_loader_768
    active_val_loader = val_loader_512 if cfg.resolution == 512 else val_loader_768

    model.train()
    running_loss, steps = 0.0, 0
    for batch in active_train_loader:
        x = batch['image'].to(accel.device)
        y_grades = batch['dr_grade'].to(accel.device)
        y_ref = batch['referable_dr'].to(accel.device)

        optimizer.zero_grad()
        with accel.autocast_context(enabled=True):
            out = model(x)
            tri = out['tri_head']
            loss_ord = coral_loss_fn(tri.ordinal_logits, y_grades)
            loss_qwk = qwk_loss_fn(tri.class_probs, y_grades, is_probs=True)
            loss_ref = F.binary_cross_entropy_with_logits(tri.referable_logit, y_ref)
            loss_cont = F.smooth_l1_loss(tri.continuous_score.squeeze(1), y_grades.float())
            total_loss = loss_ord + cfg.soft_qwk_weight * loss_qwk + 0.50 * loss_ref + 0.40 * loss_cont

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        accel.optimizer_step(optimizer)

        running_loss += float(total_loss.detach().cpu().item())
        steps += 1

    train_loss = running_loss / max(steps, 1)
    val_clin, val_arrays = evaluate_stage2(active_val_loader, qwk_weight=cfg.soft_qwk_weight)
    elapsed = time.time() - t0

    improved = ''
    if val_clin['qwk'] > best_val_qwk:
        best_val_qwk = val_clin['qwk']
        accel.save_checkpoint(model, best_ckpt_local)
        if best_ckpt_drive != best_ckpt_local:
            shutil.copy2(best_ckpt_local, best_ckpt_drive)
        np.savez_compressed(
            DRIVE_CKPT_DIR / 'stage2_val_oof.npz',
            probs=val_arrays['probs'],
            continuous_scores=val_arrays['continuous_scores'],
            referable_probs=val_arrays['referable_probs'],
            targets=val_arrays['targets'],
        )
        improved = ' * [SAVED BEST TO DRIVE]'

    print(
        f"Epoch [{epoch:02d}/{curriculum.total_epochs:02d}] ({cfg.phase}, {elapsed:.1f}s) | "
        f"Train Loss: {train_loss:.4f} | Val QWK: {val_clin['qwk']:.4f} | "
        f"Macro-F1: {val_clin['macro_f1']:.4f} | Mild-DR Recall: {val_clin['mild_dr_recall']:.3f} | "
        f"Ref-AUC: {val_clin['referable_auc']:.4f}{improved}"
    )

In [ ]:
# 5. Final Evaluation on Untouched 768x768 Test Split (Fold 0)
state_dict = torch.load(best_ckpt_local, map_location=accel.device, weights_only=True)
model.load_state_dict(state_dict)
test_clin, _ = evaluate_stage2(test_loader_768)

print('=== Stage-2 Untouched Test Split (Fold 0 @ 768x768) Results ===')
for k, v in test_clin.items():
    print(f'  {k:<32}: {v}')
print(f'Saved Checkpoint: {best_ckpt_drive}')